# Δh analysis — what does a *successful* edit look like in latent space?

**Direction:** `research/directions/delta-h-analysis.md` · `[in-frame]` · sub-Q 3 (editability). **Branch:**
`delta_h_analysis`. **Models:** GRU **and** RSSM. **No retraining** except the small Δh predictors in §5.

## Why this notebook exists

Across the whole editability thread exactly **two** edit mechanisms reliably work, and **both require oracle access**:

1. **Counterfactual state overwrite** — invent a history in which the object *always* travelled through the target,
   render it, teacher-force the model along it, and overwrite the pre-edit state with the result.
2. **Freeze-time teacher forcing** — freeze the world and teacher-force ~8 rendered frames in which the object
   interpolates from its pre-edit position to the target, then resume.

Every *direct latent write* we have tried — readout injection, Global-PCA projection, PCA geodesic, MLP-probe
gradient — fails. Note what the two survivors have in common, and it is the through-line of this whole thread:

> **We have never found an edit that works without dynamics.** Every successful mechanism operates by making the
> model **consume observations over time**. None of them writes to `h` directly.

Because these two succeed, they hand us something we have never had: **ground truth for what a successful edit
*is*, as a displacement in latent space.** Define

$$\Delta h_{\text{true}} \;=\; h_{\text{post-edit}} \;-\; h_{\text{pre-edit}}$$

and we can finally ask *why* the direct writes miss. This notebook characterises Δh: where it lives relative to the
probe, whether the two oracles agree on it, how big it is, how consistent it is across edits, and whether it can be
learned.

## The one framing that makes this sharp

The readout-injection editor produces, by construction,

$$\Delta h_{\text{pinv}} \;=\; A^{+}\,(\text{target} - (A h + b)) \;\in\; \operatorname{row}(A)$$

It can **only** ever move inside the row space of the probe. So the fraction of Δh_true lying in `row(A)` is not a
descriptive statistic — it is the **hard ceiling on how much of a successful edit that editor could achieve**, and
`‖P_row Δh‖ / ‖Δh‖` is exactly the best cosine alignment any injection-style edit could reach. That single number
turns "readable ≠ controllable" from an observation into a measurement.

## Definitions — read this before any number

### The states (all at the edit frame `ef = 20`)

| symbol | how it is built | what it represents |
|---|---|---|
| `h0` | teacher-force the **real, noisy** observations `obs[0..ef−1]` | the pre-edit state: the model's belief about frame `ef` *without* having seen the teleport |
| `h_cf` | teacher-force **clean renders of a counterfactual history** `0..ef−1` in which the edited object travelled at constant velocity *through the target*, the other object on its true path | "the state the model would have had if the world had always been headed for the target" |
| `h_ft` | from `h0`, teacher-force **N = 8 noise-matched rendered frames** interpolating the edited object pre-edit → target (other object held at its `ef` position) | "the state after the model watches the object walk to the target" |
| `h_cf_ctrl` | as `h_cf` but rendering the **true** history | *control*: isolates "clean render vs noisy observation", which otherwise contaminates `Δh_cf` |
| `h_ft_ctrl` | as `h_ft` but the object is **held at its pre-edit position** for all N frames | *control*: isolates "8 frames of frozen-world dynamics", which otherwise contaminates `Δh_ft` |

### The displacements

| name | formula | note |
|---|---|---|
| **Δh_cf** (raw) | `h_cf − h0` | the full displacement, dynamics and all |
| **Δh_ft** (raw) | `h_ft − h0` | ditto |
| **Δh_cf edit-only** | `h_cf − h_cf_ctrl` | the counterfactual *minus* the rendering change |
| **Δh_ft edit-only** | `h_ft − h_ft_ctrl` | the interpolation *minus* 8 steps of frozen dynamics |
| **Δh_pinv** | `A⁺(target − (A h0 + b))` | what readout injection actually does — the failing editor, for comparison |

Both raw and edit-only are reported throughout. The raw version is the honest "difference between a state we know
gives the outcome and where we started"; the edit-only version answers "how much of that is the edit itself".

### The measurements, and why each formula is the right one

**Row-space fraction** `f = ‖P_row(A)·Δh‖ / ‖Δh‖`, where `P_row(A) = A⁺A` projects onto the row space of the probe.

*Why this formula.* Any linear probe `p = A h + b` is blind to everything in the **null space** of `A`: adding a
null-space vector to `h` changes nothing the probe reads. The complement — the **row space**, spanned by the rows of
`A` and of dimension `d` = number of probe outputs — is the only part of `h` the probe can see *or move*. Splitting
`Δh` into these two orthogonal pieces and asking what share sits in the row space asks exactly: *how much of a
successful edit is even visible to this probe?* Because `f` is a ratio of norms of orthogonal projections, it is
also the cosine between `Δh` and its own best row-space approximation — i.e. **the largest cosine any
injection-style edit could achieve with the truth**.

*Why a baseline is mandatory.* The row space is `d` of `H` dimensions — 4 of 256 for a position probe. A **random**
vector already has `√(d/H)` = √(4/256) = **0.125** of its norm there, purely by dimension counting. So `f = 0.15`
is *not* "15% aligned, mostly missing" — it is barely above chance. Every `f` below is reported against `√(d/H)`.

**Cosine alignment** `cos(u,v) = ⟨u,v⟩ / (‖u‖‖v‖)` — the dot product of the unit vectors.

*Computed per instance, then averaged.* This matters: averaging the **vectors** first and taking one cosine of the
means would measure whether the two methods agree *on average*, washing out precisely the per-edit structure we
want. We want "do the two oracles agree on **this** edit", averaged — so every cosine below is per-sample first.

*Why a baseline is mandatory here too.* In `H` dimensions two random vectors are nearly orthogonal. Be careful
about *which* statistic `1/√H` is: for random unit vectors the **mean** cosine is **0**, and `1/√H` ≈ 0.06 (H=256)
is the **per-pair standard deviation** — the scale of chance fluctuation for a *single* pair, not a floor the mean
should sit at. So a mean cosine of +0.01 over thousands of pairs means "no shared direction", and the honest
reference for a mean is an **empirical shuffled-pair control** (cosine between Δh's belonging to *different* edits),
which is what we compare against. `1/√H` is quoted only to indicate per-pair scale.

**Magnitude ratios** — three, because "magnitude ratio" is ambiguous and each answers a different question:

| name | formula | question |
|---|---|---|
| relative-to-state | `‖Δh‖ / ‖h0‖` | how big is the edit compared with the state itself? |
| relative-to-injection | `‖Δh_true‖ / ‖Δh_pinv‖` | how much bigger is the real edit than what the failing editor attempts? |
| **relative-to-dynamics** | `‖Δh‖ / mean_t‖h_t − h_{t−1}‖` | **is a successful edit a normal-sized move, or far outside the range of ordinary dynamics?** |

**Direction consistency** — mean **pairwise** cosine over all sample pairs, against the shuffled baseline; plus the
coefficient of variation of `‖Δh‖`. Two numbers: does the oracle edit move in a *consistent direction*, and is its
size stable?

**Edit Index** and the rest of the §4 suite are the canonical set from `METRICS_AND_EDITORS.md` §4, implemented in
`scripts/editability_metrics.py`. **+1** = the output *is* the world where the edit happened, **−1** = the world
where it did not, **0** = equidistant from both. Read against each model's own unsteered row.

> **The ±1 decode convention, handled explicitly.** The GRU decoder **predicts the next** frame; the RSSM decoder
> **reconstructs the current** one. So the same teacher-forcing gives states one frame apart. Everything here is
> aligned so that **rollout step 0 decodes sim frame `ef`** for both: the GRU warms on `obs[0..ef−1]`; the RSSM
> warms on `obs[0..ef−1]` and then takes **one prior/imagine step**. Cell [1] verifies this by measurement rather
> than assertion. (`00_master_editability` accepted the offset as a known soft spot; for a study *of* Δh it has to
> be right, since a one-frame misalignment is itself a displacement.)

In [ ]:
# [1] Setup: load GRU + RSSM, the edits split, and fix the ±1 alignment BY MEASUREMENT (not assertion).
import os, sys, json, time
sys.path.insert(0, "../../..")
sys.path.insert(0, "../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, K_ROLL, N_FT = 2, 15, 8          # rollout steps ; freeze-time interpolation frames
N_EVAL, N_LEARN = 256, 1500             # analysis samples ; Δh-predictor training samples (disjoint)
OUT = "/tmp/delta_h_analysis"; os.makedirs(OUT, exist_ok=True)

MODELS = {}
MODELS["GRU"],  _ = load_checkpoint("../../../runs/controls/H256/best_model.pt", device=DEVICE)
MODELS["RSSM"], _ = load_checkpoint("../../../runs/rssm/4_dset4_refined_best/best_model.pt", device=DEVICE)
if hasattr(MODELS["RSSM"], "sample"): MODELS["RSSM"].sample = False   # deterministic posterior/prior mean
IS_RSSM = {m: hasattr(MODELS[m], "imagine_step") for m in MODELS}

bundle = load_dataset("../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res
with h5py.File(edits.h5_path, "r") as f:
    VEL_ALL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

@torch.no_grad()
def warm(model, obs_np, upto):
    """Teacher-force obs[0..upto-1] and return the flat state ALIGNED so decode(state) ↔ frame `upto`.

    GRU: its decoder already predicts the NEXT frame, so the state after obs[upto-1] is already aligned.
    RSSM: its decoder reconstructs the CURRENT frame, so one extra prior (imagine) step is needed."""
    o = torch.from_numpy(obs_np).float().to(DEVICE); state = None
    for t in range(upto):
        _, state = model.step(o[:, t], state)
    if hasattr(model, "imagine_step"):
        state, _ = model.imagine_step(state)
    return model.flat_state(state)

@torch.no_grad()
def roll(model, h_flat, steps=K_ROLL):
    st = model.state_from_flat(h_flat); out = [model.decode(st)]
    for _ in range(steps - 1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

# ── alignment CHECK, on ORDINARY (non-edit) sequences ──────────────────────────
# It must be run on the TEST split, not the edits split: on an edit sequence the pre-edit state
# legitimately fails to predict frame `ef` (the teleport is exactly what it has not seen), so the
# check would be confounded by the very effect we are studying.
obs_ev = edits.obs[:N_EVAL].astype(np.float32)
print("alignment check on ORDINARY test sequences — RMSE(decode(warm to t)), clean_obs[t+k]); min must be k=0:")
_t_chk = ef
for name, m in MODELS.items():
    hc = warm(m, test.obs[:256].astype(np.float32), _t_chk)
    dc = m.decode(m.state_from_flat(hc)).cpu().numpy()
    errs = {k: float(np.sqrt(((dc - test.clean_obs[:256, _t_chk+k])**2).mean())) for k in (-1, 0, 1)}
    best = min(errs, key=errs.get)
    print(f"  {name:<5s} k=-1 {errs[-1]:.4f} | k=0 {errs[0]:.4f} | k=+1 {errs[1]:.4f}  -> min at k={best} "
          f"{'PASS' if best == 0 else 'FAIL'}")
H0 = {name: warm(m, obs_ev, ef) for name, m in MODELS.items()}
print(f"\nmodels: GRU H={MODELS['GRU'].hidden_size}, RSSM H={MODELS['RSSM'].hidden_size} "
      f"(det {MODELS['RSSM'].cfg.det_size} + stoch {MODELS['RSSM'].hidden_size - MODELS['RSSM'].cfg.det_size})")
print(f"edits: analysis on [0,{N_EVAL}) | Δh-predictor training on [2000,{2000+N_LEARN}) (disjoint) | ef={ef}")

In [ ]:
# [2] Build the two ORACLE edit states, their dynamics/rendering controls, and the readout-injection edit.
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, np.float32)
COL  = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ, 1))
def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)
def render_traj(pos_seq, noise=0.0):
    _, _, inten = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                     colors=COL, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return inten.astype(np.float32)

OBS_NOISE = float(sim["obs_noise_std"])   # teacher-forced frames are noise-matched to training

def build_sequences(idx, ed=None, vel=None, target_override=None):
    """Counterfactual + freeze-time observation sequences (and their controls) for edit samples `idx`.

    `ed`/`vel` default to the main edits split; pass them explicitly to run on another dataset.
    `target_override` (n, 2) replaces the edited object's target position — used by §6 to hold the
    DISPLACEMENT fixed while the starting state varies."""
    ed = edits if ed is None else ed
    vel = VEL_ALL if vel is None else vel
    n = len(idx); DT = float(sim["dt"])
    oe = ed.edit_object[idx].astype(int)
    pos = ed.positions[idx][:, :, :N_OBJ, :].astype(np.float32)
    tgt = pos[:, ef].copy()                                     # edited object already at the target
    if target_override is not None:
        tgt[np.arange(n), oe] = target_override.astype(np.float32)
    pre = pos[:, ef-1]
    cf_obs  = np.zeros((n, ef, R), np.float32)                  # counterfactual history, frames 0..ef-1
    tr_obs  = np.zeros((n, ef, R), np.float32)                  # SAME rendering of the TRUE history (control)
    ft_obs  = np.zeros((n, N_FT, R), np.float32)                # freeze-time interpolation frames
    ft_ctrl = np.zeros((n, N_FT, R), np.float32)                # frozen at the PRE-EDIT position (control)
    t_idx = np.arange(ef)
    for i in range(n):
        o, other = oe[i], 1 - oe[i]
        v = vel[idx[i], ef, o]
        # counterfactual: edited object on a constant-velocity line that ARRIVES at the target at ef
        cf = np.zeros((ef, N_OBJ, 2), np.float32)
        cf[:, o]     = tgt[i, o][None, :] - v[None, :] * (ef - t_idx)[:, None] * DT
        cf[:, other] = pos[i, :ef, other]
        cf_obs[i] = render_traj(cf)
        tr_obs[i] = render_traj(pos[i, :ef])                    # true history, same clean renderer
        # freeze-time: interpolate pre -> target over N_FT frames, other object held at its ef position
        fr = np.zeros((N_FT, N_OBJ, 2), np.float32); fc = np.zeros((N_FT, N_OBJ, 2), np.float32)
        for j in range(N_FT):
            fr[j, o] = pre[i, o] + ((j+1)/N_FT) * (tgt[i, o] - pre[i, o]); fr[j, other] = tgt[i, other]
            fc[j, o] = pre[i, o];                                fc[j, other] = tgt[i, other]
        ft_obs[i]  = render_traj(fr, OBS_NOISE)
        ft_ctrl[i] = render_traj(fc, OBS_NOISE)
    return dict(cf=cf_obs, tr=tr_obs, ft=ft_obs, ft_ctrl=ft_ctrl, oe=oe, tgt=tgt, pre=pre)

@torch.no_grad()
def continue_from(model, h_flat, frames):
    """Teacher-force extra frames starting from a flat state; returns the new aligned flat state."""
    st = model.state_from_flat(h_flat)
    o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = model.step(o[:, t], st)
    if hasattr(model, "imagine_step"):
        st, _ = model.imagine_step(st)
    return model.flat_state(st)

t0 = time.perf_counter()
IDX = np.arange(N_EVAL)
SEQ = build_sequences(IDX)
print(f"rendered {N_EVAL} counterfactual + freeze-time sequences in {time.perf_counter()-t0:.0f}s")

STATES = {}
for name, m in MODELS.items():
    h0 = H0[name]
    STATES[name] = {
        "h0":        h0,
        "h_cf":      warm(m, SEQ["cf"], ef),                      # counterfactual state overwrite
        "h_cf_ctrl": warm(m, SEQ["tr"], ef),                      # same renderer, TRUE history
        "h_ft":      continue_from(m, h0, SEQ["ft"]),             # freeze-time teacher forcing
        "h_ft_ctrl": continue_from(m, h0, SEQ["ft_ctrl"]),        # frozen at pre-edit position
    }
print("built states:", {k: tuple(v["h0"].shape) for k, v in STATES.items()})

In [ ]:
# [3] The linear probes (position, and position+velocity), the readout-injection edit, and the ray zones.
@torch.no_grad()
def aligned_bank(model, obs_np):
    """States aligned exactly like the ones we edit: decode(bank[:, t]) ↔ sim frame t+1.

    This matters for the RSSM: `observe_sequence` returns POSTERIOR states (which reconstruct the
    current frame), but the state we edit is one prior step ahead. Fitting the probe on posteriors and
    then applying it to a prior state would be a one-frame mismatch — small for the GRU, but it is
    exactly the kind of offset a Δh study cannot afford."""
    o = torch.from_numpy(obs_np).float().to(DEVICE)
    T = o.shape[1]; state = None; out = []
    for t in range(T - 1):
        _, state = model.step(o[:, t], state)
        s_out = state
        if hasattr(model, "imagine_step"):
            s_out, _ = model.imagine_step(state)     # posterior -> prior (predict-next alignment)
        out.append(model.flat_state(s_out))
    return torch.stack(out, 1)                        # (N, T-1, H), decode(out[:, t]) ↔ frame t+1

def fit_linear_probe(model, obs_np, y_np):
    """Least-squares probe h -> y on ALIGNED states. Returns (A, b, A_pinv, P_row, rmse, r2)."""
    Hs = aligned_bank(model, obs_np).cpu().numpy()
    T = Hs.shape[1]
    X = Hs.reshape(-1, Hs.shape[-1]); Y = y_np[:, 1:1+T].reshape(len(X), -1)   # frame t+1
    Aug = np.concatenate([X, np.ones((len(X), 1), np.float32)], 1)
    sol, *_ = np.linalg.lstsq(Aug, Y, rcond=None)
    A = sol[:-1].T.astype(np.float32); b = sol[-1].astype(np.float32)       # A: (d, H)
    pred = X @ sol[:-1] + sol[-1]
    rmse = float(np.sqrt(((pred - Y)**2).mean()))
    r2 = float(1 - ((pred - Y)**2).sum() / ((Y - Y.mean(0))**2).sum())
    A_t = torch.tensor(A, device=DEVICE); b_t = torch.tensor(b, device=DEVICE)
    A_pinv = torch.tensor(np.linalg.pinv(A), device=DEVICE)                  # (H, d)
    P_row = A_pinv @ A_t                                                     # (H, H) projector onto row(A)
    return dict(A=A_t, b=b_t, A_pinv=A_pinv, P_row=P_row, rmse=rmse, r2=r2, d=A.shape[0])

NPROBE = 600
obs_pr = test.obs[:NPROBE].astype(np.float32)
pos_pr = test.positions[:NPROBE, :, :N_OBJ, :].reshape(NPROBE, -1, N_OBJ*2)
with h5py.File(test.h5_path, "r") as f:
    vel_pr = f["velocities"][:NPROBE, :, :N_OBJ, :].astype(np.float32).reshape(NPROBE, -1, N_OBJ*2)
posvel_pr = np.concatenate([pos_pr, vel_pr], -1)

PROBES = {}
for name, m in MODELS.items():
    PROBES[name] = {"position (d=4)":          fit_linear_probe(m, obs_pr, pos_pr),
                    "position+velocity (d=8)": fit_linear_probe(m, obs_pr, posvel_pr)}
    for pn, p in PROBES[name].items():
        print(f"  {name:<5s} {pn:<26s} d={p['d']:<2d} R²={p['r2']:.3f}  RMSE={p['rmse']:.3f} sim-units")

# readout injection — what the FAILING editor does, in row(A) by construction
tgt4 = torch.from_numpy(SEQ["tgt"].reshape(N_EVAL, N_OBJ*2)).float().to(DEVICE)
for name, m in MODELS.items():
    P = PROBES[name]["position (d=4)"]
    h0 = STATES[name]["h0"]
    STATES[name]["h_pinv"] = h0 + (tgt4 - (h0 @ P["A"].T + P["b"])) @ P["A_pinv"].T

# canonical §4 ray zones + both ground-truth worlds
gt_roll = edits.clean_obs[:N_EVAL, ef:ef+K_ROLL, :].astype(np.float32)
ZONES = build_edit_zones(pre_pos=SEQ["pre"], tgt_pos=SEQ["tgt"], pre_vel=VEL_ALL[IDX, ef-1, :N_OBJ, :],
                         edit_object=SEQ["oe"], sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N_EVAL, ef:ef+K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
print(f"\nray zones/sample: target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"differing {ZONES.differing.sum(1).mean():.1f}")

---
## §1 — Do the two oracle edits actually succeed?

Before analysing Δh we have to establish that the states it is built from are worth analysing. Both oracles are
scored on the canonical §4 set, at the edit frame **and across the whole rollout** — landing an edit and *holding*
it are different things (the decoder-gradient oracle scores +0.94 at step 0 and decays to −0.12 by step 14).

A prediction worth stating first: counterfactual overwrite **cannot revert**, because it replaces the state
outright — there is no remnant of the pre-edit belief to fall back to. Freeze-time keeps the real history in the
recurrence, so it *could* in principle be pulled back.

In [ ]:
# [4] Fig 1 — Edit Index at the edit frame and across the rollout, both oracles, both models.
CARDS = {}
for name, m in MODELS.items():
    CARDS[name] = {}
    for lab, key in [("unsteered (no edit)", "h0"), ("counterfactual state overwrite", "h_cf"),
                     ("freeze-time teacher forcing", "h_ft"), ("readout injection (the failing editor)", "h_pinv")]:
        c = edit_scorecard(roll(m, STATES[name][key]), ZONES, gt_roll)
        c["fidelity_ratio"] = fidelity_ratio(c, CARDS[name]["unsteered (no edit)"]) if lab != "unsteered (no edit)" else 1.0
        CARDS[name][lab] = c

ORDER = ["unsteered (no edit)", "readout injection (the failing editor)",
         "freeze-time teacher forcing", "counterfactual state overwrite"]
COLR = {"unsteered (no edit)": "0.55", "readout injection (the failing editor)": "#D55E00",
        "freeze-time teacher forcing": "#0072B2", "counterfactual state overwrite": "#009E73"}
rows = ["| model | edit | Edit Index (step 0) ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity |",
        "|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    for lab in ORDER:
        c = CARDS[name][lab]
        rows.append(f"| {name} | {lab} | **{c['edit_index']:+.2f}** | {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | "
                    f"{c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 1 — do the oracle edits succeed?** Edit Index: +1 = the output *is* the world where the "
                 f"edit happened, −1 = the world where it did not, 0 = equidistant. N={N_EVAL} held-out edits.\n\n"
                 + "\n".join(rows)))

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.4))
xi = np.arange(len(ORDER)); w = 0.38
for k, name in enumerate(MODELS):
    ax[0].bar(xi + (k-0.5)*w, [CARDS[name][l]["edit_index"] for l in ORDER], w,
              color=["#0072B2", "#E69F00"][k], label=name)
for y, lab in [(1.0, "edited world"), (0.0, "equidistant"), (-1.0, "unedited world")]:
    ax[0].axhline(y, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(lab, xy=(len(ORDER)-0.4, y), fontsize=7.5, color="0.35", ha="right", va="bottom")
ax[0].set_xticks(xi); ax[0].set_xticklabels(ORDER, fontsize=8, rotation=25, ha="right")
ax[0].set_ylim(-1.05, 1.05); ax[0].set_ylabel("Edit Index"); ax[0].legend(fontsize=8)
ax[0].set_title("(a) does the edit land? (step 0)", fontsize=10); ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])
for j, name in enumerate(MODELS):
    a = ax[1+j]; s = np.arange(K_ROLL)
    for lab in ORDER:
        a.plot(s, CARDS[name][lab]["edit_index_by_step"], color=COLR[lab], lw=2.0, label=lab)
    a.axhline(0, color="0.4", ls=":", lw=1.0); a.set_ylim(-1.05, 1.05)
    a.set_xlabel("rollout step s (0 = sim frame ef)"); a.set_ylabel("Edit Index")
    a.set_title(f"({'bc'[j]}) does it HOLD? — {name}", fontsize=10); a.grid(alpha=0.3); style_ax(a)
ax[1].legend(fontsize=7.5, loc="lower left")
fig.suptitle("Fig 1 — the two oracle edits, on the canonical Edit Index", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_edit_index.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §2 — Where does Δh live relative to the probe?

The core measurement. For each edit sample we split `Δh` into the part the position probe can see and move
(`row(A)`, 4 dims) and the part it is blind to (`null(A)`, 252 dims), and report
`f = ‖P_row·Δh‖ / ‖Δh‖` — **per instance, then averaged**.

Remember the chance level: a random vector already scores `√(d/H)`. Anything at or below that line means the
successful edit is, as far as the probe is concerned, **indistinguishable from a random direction**.

In [ ]:
# [5] Fig 2 + Table 2 — row-space fraction of Δh, per instance then averaged, against the chance level.
def row_frac(dh, P_row):
    """Per-sample ‖P_row·Δh‖ / ‖Δh‖ — the largest cosine any injection-style edit could reach with Δh."""
    num = torch.linalg.norm(dh @ P_row.T, dim=-1)
    den = torch.linalg.norm(dh, dim=-1).clamp_min(1e-9)
    return (num / den).cpu().numpy()

DH = {}
for name in MODELS:
    S = STATES[name]
    DH[name] = {
        "counterfactual — raw":        S["h_cf"] - S["h0"],
        "counterfactual — edit-only":  S["h_cf"] - S["h_cf_ctrl"],
        "freeze-time — raw":           S["h_ft"] - S["h0"],
        "freeze-time — edit-only":     S["h_ft"] - S["h_ft_ctrl"],
        "readout injection (failing)": S["h_pinv"] - S["h0"],
    }

RF = {}
rows = ["| model | Δh | probe | row-space fraction `f` | chance `√(d/H)` | enrichment ×chance |",
        "|---|---|---|---|---|---|"]
for name in MODELS:
    RF[name] = {}
    Hdim = MODELS[name].hidden_size
    for dn, dh in DH[name].items():
        RF[name][dn] = {}
        for pn, P in PROBES[name].items():
            f = row_frac(dh, P["P_row"]); chance = np.sqrt(P["d"]/Hdim)
            RF[name][dn][pn] = (float(f.mean()), float(f.std()), chance)
            rows.append(f"| {name} | {dn} | {pn} | **{f.mean():.3f}** ± {f.std():.3f} | {chance:.3f} | "
                        f"{f.mean()/chance:.1f}× |")
display(Markdown("**Table 2 — how much of a successful edit can a linear probe even see?** `f` is computed per "
                 "sample then averaged (± sd across samples). `f` is also the **ceiling on the cosine** any "
                 "injection-style editor could achieve with that Δh.\n\n" + "\n".join(rows)))

plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(15.5, 4.6), sharey=True)
dnames = list(DH["GRU"].keys()); xi = np.arange(len(dnames)); w = 0.38
for j, name in enumerate(MODELS):
    a = axes[j]
    for k, pn in enumerate(PROBES[name]):
        vals = [RF[name][d][pn][0] for d in dnames]
        errs = [RF[name][d][pn][1] for d in dnames]
        a.bar(xi + (k-0.5)*w, vals, w, yerr=errs, capsize=2, color=["#0072B2", "#E69F00"][k], label=pn)
        ch = RF[name][dnames[0]][pn][2]
        a.axhline(ch, color=["#0072B2", "#E69F00"][k], ls=":", lw=1.4)
        a.annotate(f"chance for {pn}: {ch:.3f}", xy=(len(dnames)-0.45, ch), fontsize=7,
                   color=["#0072B2", "#E69F00"][k], ha="right", va="bottom")
    a.set_xticks(xi); a.set_xticklabels(dnames, fontsize=8, rotation=25, ha="right")
    a.set_title(f"{name}", fontsize=11); a.grid(alpha=0.3, axis="y"); style_ax(a)
    a.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("row-space fraction  ‖P_row·Δh‖ / ‖Δh‖")
fig.suptitle("Fig 2 — the reachability ceiling: what share of a successful edit lies in the probe's row space "
             "(dotted = chance for a random vector)", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_row_space.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §3 — Do the two oracles agree, how big is Δh, and is its direction consistent?

Three questions, one figure. All cosines are **per instance, then averaged**, against two baselines: the
theoretical `1/√H` for random vectors, and a **shuffled-pair** control (cosine between Δh's from *different* edit
samples), which keeps the real distribution of Δh and is the stricter test.

If the two oracles' Δh's are strongly aligned, there is a canonical "move the object here" direction and an edit
map is a learnable object. If they are near the shuffled baseline, then many different latent states render the
same target scene — which would explain both why probe-directed writes miss and why a learned editor plateaus.

In [ ]:
# [6] Fig 3 + Table 3 — pairwise cosines, magnitude ratios, and direction consistency.
def cos_per_instance(u, v):
    return torch.nn.functional.cosine_similarity(u, v, dim=-1).cpu().numpy()

def mean_pairwise_cos(dh, n_pairs=20000, seed=0):
    """Mean cosine over RANDOM PAIRS of samples — 'does the oracle edit move in a consistent direction?'"""
    g = np.random.default_rng(seed); n = len(dh)
    i = g.integers(0, n, n_pairs); j = g.integers(0, n, n_pairs); k = i != j
    u = torch.nn.functional.normalize(dh, dim=-1)
    return float((u[i[k]] * u[j[k]]).sum(-1).mean())

# per-step latent displacement during ORDINARY dynamics — the reference scale for "how big is this edit"
STEP_NORM = {}
for name, m in MODELS.items():
    Hs = aligned_bank(m, obs_ev)
    STEP_NORM[name] = float(torch.linalg.norm(Hs[:, 1:] - Hs[:, :-1], dim=-1).mean())

ORACLES = ["counterfactual — raw", "counterfactual — edit-only", "freeze-time — raw", "freeze-time — edit-only"]
COS, MAG = {}, {}
rows = ["| model | quantity | value | baseline | reading |", "|---|---|---|---|---|"]
for name in MODELS:
    Hdim = MODELS[name].hidden_size; rnd = 1/np.sqrt(Hdim)
    d_cf, d_ft, d_pv = DH[name]["counterfactual — raw"], DH[name]["freeze-time — raw"], DH[name]["readout injection (failing)"]
    d_cfe, d_fte = DH[name]["counterfactual — edit-only"], DH[name]["freeze-time — edit-only"]
    shuf = np.random.default_rng(0).permutation(len(d_cf))
    COS[name] = {
        "counterfactual vs freeze-time (raw)":       float(cos_per_instance(d_cf, d_ft).mean()),
        "counterfactual vs freeze-time (edit-only)": float(cos_per_instance(d_cfe, d_fte).mean()),
        "counterfactual vs readout injection":       float(cos_per_instance(d_cf, d_pv).mean()),
        "freeze-time vs readout injection":          float(cos_per_instance(d_ft, d_pv).mean()),
        "SHUFFLED control (cf vs ft, mismatched)":   float(cos_per_instance(d_cf, d_ft[shuf]).mean()),
    }
    for q, v in COS[name].items():
        rows.append(f"| {name} | cosine: {q} | **{v:+.3f}** | random {rnd:+.3f} | "
                    f"{'aligned' if v > 5*rnd else 'near-orthogonal'} |")
    MAG[name] = {}
    for dn in ORACLES:
        dh = DH[name][dn]
        MAG[name][dn] = {
            "‖Δh‖ / ‖h0‖":            float((torch.linalg.norm(dh, dim=-1)/torch.linalg.norm(STATES[name]['h0'], dim=-1)).mean()),
            "‖Δh_true‖ / ‖Δh_pinv‖":  float((torch.linalg.norm(dh, dim=-1)/torch.linalg.norm(d_pv, dim=-1).clamp_min(1e-9)).mean()),
            "‖Δh‖ / mean‖h_t−h_{t−1}‖": float(torch.linalg.norm(dh, dim=-1).mean())/STEP_NORM[name],
            "direction consistency (mean pairwise cos)": mean_pairwise_cos(dh),
            "magnitude CV": float(torch.linalg.norm(dh, dim=-1).std()/torch.linalg.norm(dh, dim=-1).mean()),
        }
display(Markdown("**Table 3 — agreement between the two oracles, and against the failing editor.**\n\n" + "\n".join(rows)))

rows2 = ["| model | Δh | ‖Δh‖/‖h0‖ | ‖Δh_true‖/‖Δh_pinv‖ | ‖Δh‖ / per-step ‖Δh_dyn‖ | direction consistency | magnitude CV |",
         "|---|---|---|---|---|---|---|"]
for name in MODELS:
    for dn in ORACLES:
        q = MAG[name][dn]
        rows2.append(f"| {name} | {dn} | {q['‖Δh‖ / ‖h0‖']:.2f} | {q['‖Δh_true‖ / ‖Δh_pinv‖']:.2f} | "
                     f"{q['‖Δh‖ / mean‖h_t−h_{t−1}‖']:.1f}× | {q['direction consistency (mean pairwise cos)']:+.3f} | "
                     f"{q['magnitude CV']:.2f} |")
display(Markdown("**Table 4 — magnitude and consistency.** *direction consistency* = mean cosine over random pairs "
                 "of edit samples (random baseline ≈ 1/√H: "
                 f"{1/np.sqrt(MODELS['GRU'].hidden_size):.3f} GRU, {1/np.sqrt(MODELS['RSSM'].hidden_size):.3f} RSSM). "
                 "*magnitude CV* = sd/mean of ‖Δh‖.\n\n" + "\n".join(rows2)))

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
cn = list(COS["GRU"].keys()); xi = np.arange(len(cn)); w = 0.38
for k, name in enumerate(MODELS):
    ax[0].bar(xi + (k-0.5)*w, [COS[name][c] for c in cn], w, color=["#0072B2", "#E69F00"][k], label=name)
    ax[0].axhline(1/np.sqrt(MODELS[name].hidden_size), color=["#0072B2", "#E69F00"][k], ls=":", lw=1.2)
ax[0].set_xticks(xi); ax[0].set_xticklabels(cn, fontsize=7.5, rotation=25, ha="right")
ax[0].set_ylabel("mean per-instance cosine"); ax[0].axhline(0, color="0.3", lw=0.8)
ax[0].set_title("(a) do the two oracles agree?\n(dotted = random-vector baseline 1/√H)", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])
xi2 = np.arange(len(ORACLES))
for k, name in enumerate(MODELS):
    ax[1].bar(xi2 + (k-0.5)*w, [MAG[name][d]["‖Δh‖ / mean‖h_t−h_{t−1}‖"] for d in ORACLES], w,
              color=["#0072B2", "#E69F00"][k], label=name)
ax[1].axhline(1.0, color="0.35", ls=":", lw=1.2)
ax[1].annotate("one ordinary dynamics step", xy=(len(ORACLES)-0.45, 1.0), fontsize=7.5, color="0.35",
               ha="right", va="bottom")
ax[1].set_xticks(xi2); ax[1].set_xticklabels(ORACLES, fontsize=8, rotation=25, ha="right")
ax[1].set_ylabel("‖Δh‖ ÷ mean per-step ‖h_t − h_{t−1}‖")
ax[1].set_title("(b) how big is a successful edit,\nin units of ordinary dynamics?", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, axis="y"); style_ax(ax[1])
for k, name in enumerate(MODELS):
    ax[2].bar(xi2 + (k-0.5)*w, [MAG[name][d]["direction consistency (mean pairwise cos)"] for d in ORACLES], w,
              color=["#0072B2", "#E69F00"][k], label=name)
    ax[2].axhline(1/np.sqrt(MODELS[name].hidden_size), color=["#0072B2", "#E69F00"][k], ls=":", lw=1.2)
ax[2].set_xticks(xi2); ax[2].set_xticklabels(ORACLES, fontsize=8, rotation=25, ha="right")
ax[2].set_ylabel("mean pairwise cosine across edits"); ax[2].axhline(0, color="0.3", lw=0.8)
ax[2].set_title("(c) is the edit direction CONSISTENT\nacross different edits?", fontsize=10)
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3, axis="y"); style_ax(ax[2])
fig.suptitle("Fig 3 — geometry of a successful edit: agreement, magnitude, consistency", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_geometry.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §4 — Row-space fraction versus probe accuracy

Your hypothesis: *when the probe is accurate, maybe Δh lies more in its row space.* Within one model the probe
accuracy barely varies, so there is nothing to plot against. The x-axis comes from the **8 GRUs trained on the
`michael_controls` branch** (`runs/controls/`: the hidden-size sweep H = 8…512 and the noise 2×2), which span
position-probe R² from **0.175 to 0.855**. Borrowed purely as a source of probe-accuracy variation.

In [ ]:
# [7] Fig 4 — does a more accurate probe capture more of the edit? (8 controls GRUs as the x-axis)
SWEEP = ["H8", "H32", "H128", "H256", "H512", "N_obs0_pos0", "N_obs0_pos004", "N_obs02_pos0"]
SW_DS = {"N_obs0_pos0": "9_obsnoise0_posnoise0", "N_obs0_pos004": "10_obsnoise0_posnoise004",
         "N_obs02_pos0": "11_obsnoise02_posnoise0"}
pts = []
for code in SWEEP:
    ck = f"../../../runs/controls/{code}/best_model.pt"
    if not os.path.exists(ck): continue
    m, _ = load_checkpoint(ck, device=DEVICE)
    ds = SW_DS.get(code, "4_fixed_refl_inview")
    if ds == "4_fixed_refl_inview":
        o_pr, p_pr, edb, vb = obs_pr, pos_pr, edits, VEL_ALL
    else:
        bb = load_dataset(f"../../../datasets/{ds}", n_obj_keep=N_OBJ); edb = bb.edits
        o_pr = bb.test.obs[:NPROBE].astype(np.float32)
        p_pr = bb.test.positions[:NPROBE, :, :N_OBJ, :].reshape(NPROBE, -1, N_OBJ*2)
        with h5py.File(edb.h5_path, "r") as f: vb = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)
    P = fit_linear_probe(m, o_pr, p_pr)
    n_s = 96
    idx = np.arange(n_s)
    sv = VEL_ALL if ds == "4_fixed_refl_inview" else vb
    # rebuild the two oracle states on this model's own dataset (no global mutation)
    sq = build_sequences(idx, ed=edb, vel=sv)
    h0b = warm(m, edb.obs[:n_s].astype(np.float32), ef)
    dh_cf = warm(m, sq["cf"], ef) - h0b
    dh_ft = continue_from(m, h0b, sq["ft"]) - h0b
    pts.append(dict(code=code, r2=P["r2"],
                    f_cf=float(row_frac(dh_cf, P["P_row"]).mean()),
                    f_ft=float(row_frac(dh_ft, P["P_row"]).mean()),
                    chance=np.sqrt(P["d"]/m.hidden_size)))
    q = pts[-1]
    print(f"  {code:<15s} probe R²={P['r2']:.3f}  chance={q['chance']:.3f}  "
          f"f(counterfactual)={q['f_cf']:.3f} ({q['f_cf']/q['chance']:.2f}x chance)  "
          f"f(freeze-time)={q['f_ft']:.3f} ({q['f_ft']/q['chance']:.2f}x chance)")

plt.style.use("default")
# chance = sqrt(d/H) varies enormously across this sweep (0.707 at H=8, 0.088 at H=512), so a single
# chance line would be meaningless — plot the ENRICHMENT f / chance instead. 1.0 = indistinguishable
# from a random direction as far as the probe is concerned.
fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.8))
r2s = [p["r2"] for p in pts]
for key, lab, col in [("f_cf", "counterfactual state overwrite", "#009E73"),
                      ("f_ft", "freeze-time teacher forcing", "#0072B2")]:
    raw = [p[key] for p in pts]; enr = [p[key]/p["chance"] for p in pts]
    ax[0].scatter(r2s, raw, s=70, color=col, edgecolor="k", linewidth=0.6, label=lab, zorder=3)
    ax[1].scatter(r2s, enr, s=70, color=col, edgecolor="k", linewidth=0.6, label=lab, zorder=3)
    for a_, ys in ((ax[0], raw), (ax[1], enr)):
        z = np.polyfit(r2s, ys, 1); xs = np.linspace(min(r2s), max(r2s), 50)
        a_.plot(xs, np.polyval(z, xs), color=col, lw=1.2, alpha=0.6)
        a_.annotate(f"r = {np.corrcoef(r2s, ys)[0,1]:+.2f}", xy=(xs[-1], np.polyval(z, xs[-1])),
                    fontsize=8, color=col, ha="right")
ax[0].scatter(r2s, [p["chance"] for p in pts], s=40, marker="_", color="0.4",
              label="chance √(d/H) for that model")
for p in pts:
    ax[0].annotate(p["code"], xy=(p["r2"], p["f_ft"]), fontsize=6.5, color="0.35",
                   xytext=(3, -9), textcoords="offset points")
    ax[1].annotate(p["code"], xy=(p["r2"], p["f_ft"]/p["chance"]), fontsize=6.5, color="0.35",
                   xytext=(3, -9), textcoords="offset points")
ax[1].axhline(1.0, color="0.35", ls=":", lw=1.4)
ax[1].annotate("indistinguishable from a random direction", xy=(max(r2s), 1.0), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[0].set_ylabel("row-space fraction of Δh  (raw)")
ax[0].set_title("(a) raw fraction — NOT comparable across models\n(chance depends on H: 0.71 at H=8, 0.09 at H=512)",
                fontsize=10)
ax[1].set_ylabel("enrichment:  row-space fraction ÷ chance")
ax[1].set_title("(b) the comparable view", fontsize=10)
for a_ in ax:
    a_.set_xlabel("linear position-probe R² (across the 8 controls GRUs)")
    a_.legend(fontsize=8); a_.grid(alpha=0.3); style_ax(a_)
fig.suptitle("Fig 4 — does a more accurate probe capture more of a successful edit?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_probe_accuracy.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §5 — Can the edit be *learned* from oracle Δh?

We now have thousands of (state, target) → Δh_true pairs from a mechanism that demonstrably works. So: fit a
predictor `g(h0, target) → Δh`, apply `h0 + g(h0, target)`, and score it with the **same Edit Index on held-out
edits**. This is a better-posed version of the amortized-editor experiment, because the predictor is trained by
**direct supervision on a known-good displacement** rather than only on a rollout loss.

Linear and MLP predictors, one per oracle source. Trained on `edits[2000:2000+1500]`, evaluated on the same
held-out `edits[:256]` used everywhere above.

In [ ]:
# [8] Fig 5 + Table 5 — learn g(h0,target) -> Δh from oracle pairs; evaluate on HELD-OUT edits.
LRN_IDX = np.arange(2000, 2000 + N_LEARN)
t0 = time.perf_counter()
SEQ_L = build_sequences(LRN_IDX)
print(f"rendered {N_LEARN} training sequences in {time.perf_counter()-t0:.0f}s")
tgt_L = torch.from_numpy(SEQ_L["tgt"].reshape(N_LEARN, N_OBJ*2)).float().to(DEVICE)

def fit_predictor(X, Y, kind, epochs=300, width=512, lr=1e-3):
    if kind == "linear":
        Xa = torch.cat([X, torch.ones(len(X), 1, device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Y).solution
        return lambda Z: torch.cat([Z, torch.ones(len(Z), 1, device=DEVICE)], 1) @ sol
    net = torch.nn.Sequential(torch.nn.Linear(X.shape[1], width), torch.nn.ReLU(),
                              torch.nn.Linear(width, width), torch.nn.ReLU(),
                              torch.nn.Linear(width, Y.shape[1])).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=DEVICE)
        for j in range(0, len(X), 256):
            k = perm[j:j+256]; opt.zero_grad()
            ((net(X[k]) - Y[k])**2).mean().backward(); opt.step()
    net.eval()
    return lambda Z: net(Z).detach()

LEARNED = {}
for name, m in MODELS.items():
    h0_L = warm(m, edits.obs[LRN_IDX].astype(np.float32), ef)
    dh_L = {"counterfactual": warm(m, SEQ_L["cf"], ef) - h0_L,
            "freeze-time":    continue_from(m, h0_L, SEQ_L["ft"]) - h0_L}
    X_tr = torch.cat([h0_L, tgt_L], 1)
    X_te = torch.cat([STATES[name]["h0"], tgt4], 1)
    # the SAME Δh on the held-out edits, so we can separate "memorised" from "learned but imprecise"
    dh_te = {"counterfactual": STATES[name]["h_cf"] - STATES[name]["h0"],
             "freeze-time":    STATES[name]["h_ft"] - STATES[name]["h0"]}
    LEARNED[name] = {}
    for src, Y in dh_L.items():
        for kind in ("linear", "MLP"):
            g = fit_predictor(X_tr, Y, kind)
            with torch.no_grad():
                pred_tr = g(X_tr); r2 = float(1 - ((pred_tr-Y)**2).sum()/((Y-Y.mean(0))**2).sum())
                Yte = dh_te[src]; pred_te = g(X_te)
                r2_te = float(1 - ((pred_te-Yte)**2).sum()/((Yte-Yte.mean(0))**2).sum())
                cos_te = float(torch.nn.functional.cosine_similarity(pred_te, Yte, dim=-1).mean())
                h_new = STATES[name]["h0"] + pred_te
            c = edit_scorecard(roll(m, h_new), ZONES, gt_roll)
            c["fidelity_ratio"] = fidelity_ratio(c, CARDS[name]["unsteered (no edit)"])
            c["train_r2"] = r2; c["test_r2"] = r2_te; c["test_cos"] = cos_te
            LEARNED[name][f"{src} · {kind}"] = c

rows = ["| model | learned Δh predictor | train R² on Δh | **held-out R² on Δh** | held-out cos(Δĥ, Δh) | Edit Index (held-out) ↑ | Ghost RMSE ↓ | GT-traj RMSE ↓ |",
        "|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    u = CARDS[name]['unsteered (no edit)']
    rows.append(f"| {name} | *unsteered (no edit)* | — | — | — | {u['edit_index']:+.2f} | "
                f"{u['ghost_rmse']:.3f} | {u['gt_traj_rmse']:.3f} |")
    for k, c in LEARNED[name].items():
        rows.append(f"| {name} | {k} | {c['train_r2']:.3f} | **{c['test_r2']:+.3f}** | {c['test_cos']:+.3f} | "
                    f"**{c['edit_index']:+.2f}** | {c['ghost_rmse']:.3f} | {c['gt_traj_rmse']:.3f} |")
    for k in ("counterfactual state overwrite", "freeze-time teacher forcing"):
        rows.append(f"| {name} | *{k} (the ORACLE it imitates)* | — | — | — | {CARDS[name][k]['edit_index']:+.2f} | "
                    f"{CARDS[name][k]['ghost_rmse']:.3f} | {CARDS[name][k]['gt_traj_rmse']:.3f} |")
display(Markdown("**Table 5 — can the edit be learned from oracle demonstrations?** Trained on 1500 disjoint edits, "
                 f"evaluated on the held-out {N_EVAL}. `train R²` is how well the predictor fits Δh itself; the "
                 "Edit Index is whether applying it actually edits.\n\n" + "\n".join(rows)))

plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(15.5, 4.6), sharey=True)
for j, name in enumerate(MODELS):
    a = axes[j]; keys = list(LEARNED[name].keys())
    labs = ["unsteered"] + keys + ["counterfactual\n(oracle)", "freeze-time\n(oracle)"]
    vals = ([CARDS[name]["unsteered (no edit)"]["edit_index"]]
            + [LEARNED[name][k]["edit_index"] for k in keys]
            + [CARDS[name]["counterfactual state overwrite"]["edit_index"],
               CARDS[name]["freeze-time teacher forcing"]["edit_index"]])
    cols = ["0.55"] + ["#0072B2", "#56B4E9", "#009E73", "#66C2A5"][:len(keys)] + ["#117733", "#332288"]
    a.bar(np.arange(len(labs)), vals, 0.62, color=cols)
    for y, lb in [(1.0, "edited world"), (0.0, "equidistant"), (-1.0, "unedited world")]:
        a.axhline(y, color="0.4", ls=":", lw=1.0)
    a.set_xticks(np.arange(len(labs))); a.set_xticklabels(labs, fontsize=7.5, rotation=25, ha="right")
    a.set_ylim(-1.05, 1.05); a.set_title(name, fontsize=11); a.grid(alpha=0.3, axis="y"); style_ax(a)
axes[0].set_ylabel("Edit Index (held-out edits)")
fig.suptitle("Fig 5 — learning the edit map from oracle Δh: does supervised imitation of a working edit work?",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_learned_delta_h.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §6 — Is Δh a function of the *displacement*, or of the whole starting state?

§5 found no shared direction across edits — but those edits are random teleports, so they are different *moves*.
The sharper question, and the one that decides whether an edit map could be simple: **hold the object's positional
change fixed and vary everything else.** If object 1 moves +1.5 right and +1.5 up, is Δh the same regardless of
where it started and what `h0` was?

If yes, the edit map is essentially a lookup table over displacements and readout injection was simply looking in
the wrong basis. If no, Δh depends on the full state and there is no displacement→Δh function to learn.

**A prediction worth stating first.** The observation is a **perspective** scan: an object at depth `y = 3`
subtends far more rays than the same object at `y = 11`, so the *same* world-space displacement produces a very
different change in the observation depending on where it happens. Δh should therefore be *displacement-dependent
but not displacement-only*, and the residual should track **depth**. Panel (b) tests exactly that.

**Construction.** For each of several canonical displacements `δ`, we take many different base sequences and
re-target the edited object to `pre_position + δ` (keeping only samples whose target stays inside the frustum, with
a one-radius margin). Everything else — the starting state, both objects' absolute positions, the history — varies
freely. Then the same two oracles produce Δh, and we ask how aligned those Δh's are **within** a displacement group.

In [ ]:
# [10] §6 — same displacement, different starting states: is Δh determined by the positional change?
XF, YF, YN, RAD_O = sim["x_far"], sim["y_far"], sim["y_near"], sim["radius"]
def in_frustum(p):
    x, y = p[..., 0], p[..., 1]
    return (y > YN + RAD_O) & (y < YF - RAD_O) & (np.abs(x) < (XF / YF) * y - RAD_O)

DELTAS = [(+1.5, +1.5), (+3.0, 0.0), (0.0, +3.0), (-1.5, +1.5), (-3.0, 0.0)]
POOL = np.arange(0, 1200)                       # a wider pool so each displacement gets enough in-frustum samples
pool_oe  = edits.edit_object[POOL].astype(int)
pool_pre = edits.positions[POOL, ef-1, :N_OBJ, :].astype(np.float32)[np.arange(len(POOL)), pool_oe]

GROUPS = {}
for d in DELTAS:
    dv = np.array(d, np.float32)
    ok = in_frustum(pool_pre + dv)
    sel = POOL[ok][:64]
    if len(sel) >= 24:
        GROUPS[f"δ = ({d[0]:+.1f}, {d[1]:+.1f})"] = (sel, dv)
print("displacement groups (same positional change, different starting states):")
for g, (sel, dv) in GROUPS.items():
    print(f"  {g}   n = {len(sel):>3d}   |δ| = {np.linalg.norm(dv):.2f} sim-units")

def group_delta_h(model, sel, dv):
    pre_o = edits.positions[sel, ef-1, :N_OBJ, :].astype(np.float32)
    oe_g  = edits.edit_object[sel].astype(int)
    tgt_o = pre_o[np.arange(len(sel)), oe_g] + dv
    sq = build_sequences(sel, target_override=tgt_o)
    h0g = warm(model, edits.obs[sel].astype(np.float32), ef)
    return {"counterfactual": warm(model, sq["cf"], ef) - h0g,
            "freeze-time":    continue_from(model, h0g, sq["ft"]) - h0g}, pre_o[np.arange(len(sel)), oe_g]

def pairwise_cos_and_depth(dh, pre_xy):
    # all within-group pairs: cosine, and |depth difference| between the two starting positions
    u = torch.nn.functional.normalize(dh, dim=-1)
    Cm = (u @ u.T).cpu().numpy(); n = len(u)
    iu = np.triu_indices(n, k=1)
    dy = np.abs(pre_xy[:, 1][:, None] - pre_xy[:, 1][None, :])
    return Cm[iu], dy[iu]

GRP = {}
for name, m in MODELS.items():
    GRP[name] = {}
    for g, (sel, dv) in GROUPS.items():
        dhs, pre_xy = group_delta_h(m, sel, dv)
        GRP[name][g] = {src: pairwise_cos_and_depth(dh, pre_xy) for src, dh in dhs.items()}

rows = ["| model | Δh source | displacement group | within-group mean pairwise cos | mixed-displacement (§5) | random 1/√H |",
        "|---|---|---|---|---|---|"]
for name in MODELS:
    rnd = 1/np.sqrt(MODELS[name].hidden_size)
    mixed = MAG[name]["counterfactual — raw"]["direction consistency (mean pairwise cos)"]
    for src in ("counterfactual", "freeze-time"):
        for g in GROUPS:
            c, _ = GRP[name][g][src]
            rows.append(f"| {name} | {src} | {g} | **{c.mean():+.3f}** | {mixed:+.3f} | {rnd:+.3f} |")
display(Markdown("**Table 6 — does the same positional change produce the same Δh?** Mean cosine over all pairs "
                 "*within* a displacement group (same δ, different starting states), against the mixed-displacement "
                 "value from §5 and the random-vector baseline.\n\n" + "\n".join(rows)))

In [ ]:
# [11] Fig 6 — within-displacement alignment, and whether the residual is explained by perspective (depth).
plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(15.5, 4.8))
gn = list(GROUPS.keys()); xi = np.arange(len(gn)); w = 0.2
for k, (name, src) in enumerate([(n, s) for n in MODELS for s in ("counterfactual", "freeze-time")]):
    vals = [GRP[name][g][src][0].mean() for g in gn]
    ax[0].bar(xi + (k-1.5)*w, vals, w, color=["#009E73", "#66C2A5", "#0072B2", "#56B4E9"][k],
              label=f"{name} · {src}")
for name, col in [("GRU", "#333333"), ("RSSM", "#888888")]:
    ax[0].axhline(MAG[name]["counterfactual — raw"]["direction consistency (mean pairwise cos)"],
                  color=col, ls="--", lw=1.3, label=f"{name}: ACROSS displacements (§5 control)")
ax[0].axhline(0.0, color="0.25", lw=1.0)
ax[0].annotate("0 = unrelated directions", xy=(len(gn)-0.55, 0.002), fontsize=7.5, color="0.3", ha="right")
ax[0].set_xticks(xi); ax[0].set_xticklabels(gn, fontsize=8, rotation=20, ha="right")
ax[0].set_ylabel("mean pairwise cosine within the group"); ax[0].axhline(0, color="0.3", lw=0.8)
ax[0].set_title("(a) same displacement, different starting states:\ndo the Δh's align?", fontsize=10)
ax[0].legend(fontsize=7, ncol=2); ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])

# (b) within a displacement group, does alignment decay as the two starts differ in DEPTH?
for name, col in [("GRU", "#0072B2"), ("RSSM", "#E69F00")]:
    cs = np.concatenate([GRP[name][g]["counterfactual"][0] for g in gn])
    ds = np.concatenate([GRP[name][g]["counterfactual"][1] for g in gn])
    bins = np.linspace(0, np.percentile(ds, 95), 9); mid = 0.5*(bins[1:]+bins[:-1])
    prof = [cs[(ds >= bins[i]) & (ds < bins[i+1])].mean() if ((ds >= bins[i]) & (ds < bins[i+1])).sum() > 5
            else np.nan for i in range(len(bins)-1)]
    ax[1].plot(mid, prof, "-o", ms=5, color=col, label=f"{name} (counterfactual Δh)")
    r = np.corrcoef(ds, cs)[0, 1]
    ax[1].annotate(f"{name}: r = {r:+.2f}", xy=(mid[-1], prof[-1] if not np.isnan(prof[-1]) else 0),
                   fontsize=8, color=col, ha="right")
ax[1].set_xlabel("|difference in the two starting DEPTHS| (sim units)")
ax[1].set_ylabel("mean pairwise cosine")
ax[1].set_title("(b) is the leftover dependence PERSPECTIVE?\nalignment vs depth mismatch, within a displacement",
                fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); style_ax(ax[1])
fig.suptitle("Fig 6 — is Δh determined by the positional change alone?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_same_displacement.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

for name in MODELS:
    wg = np.mean([GRP[name][g]["counterfactual"][0].mean() for g in gn])
    mx = MAG[name]["counterfactual — raw"]["direction consistency (mean pairwise cos)"]
    rnd = 1/np.sqrt(MODELS[name].hidden_size)
    print(f"{name}: mean pairwise cosine WITHIN a fixed displacement {wg:+.3f} vs ACROSS displacements {mx:+.3f} "
          f"(expected 0 for unrelated directions; per-pair sd {rnd:.3f})")
    print(f"        -> holding the displacement fixed multiplies alignment by {wg/max(mx,1e-6):.1f}x, but the "
          f"absolute level is {wg:.3f}: "
          + ("the displacement DETERMINES the direction" if wg > 0.5 else
             "the displacement carries real information about Δh, yet is nowhere near determining it — "
             "Δh still depends mostly on the rest of the state"))

---
## §7 — Compositionality: do edits add up?

Two questions, and it matters that they are different.

**(a) Sequential composition — is the latent path-independent?** Move the object `p0 → p1 → p2` in two freeze-time
steps, versus `p0 → p2` in one. Both end with the object in the same place. If the latent were a function of the
current world configuration the two states would coincide; if it accumulates the path, they will not.

> **The waypoint must be off the straight line.** With `p1` at the *midpoint*, an 8+8-frame two-step route
> traverses **exactly** the same frames as a 16-frame direct interpolation (verified: maximum difference 0.0), so
> the model sees an identical observation sequence and the test is vacuous by construction. `p1` is therefore the
> midpoint offset **2 sim-units perpendicular** to `p0 → p2`: same start, same endpoint, genuinely different path.

> **Only freeze-time can test this.** Counterfactual overwrite *replaces the entire history* with a fabricated one
> that arrives at the target, so `CF(anything, p2) = CF(p2)` — it is path-independent **by construction** and would
> "pass" while measuring nothing. Freeze-time keeps the real history and appends frames, so it is the mechanism at
> risk. **Frame counts are matched**: direct uses 16 interpolation frames, the two-step route 8 + 8, so the only
> difference between them is the *path*, not how long the model spent absorbing frames.

**(b) Object superposition — is the map additively separable across objects?**

$$[\,\text{move obj0}\,] \;+\; [\,\text{move obj1}\,] \;\;\overset{?}{=}\;\; [\,\text{move both}\,]$$

If the latent genuinely factored into objects, per-object edits would be independent displacements that add. This is
the sharpest *object-handle* test available: it asks about factorisation directly rather than about reachability.

*The **state-space** version of this test is not tautological for either mechanism* — each state comes from an
independent render → teacher-force run, so nothing algebraically forces the displacements to add. **The
decode-level version is a different matter**: with a linear decoder, the composed state's rendered output is forced
regardless of any structure in the latent, so the Edit Index column below cannot be read as independent evidence.
**§7b quantifies exactly this and is required reading for Table 7.** We run it with **both** oracles, because they ask
different things: counterfactual overwrite is memoryless, so it isolates whether the **configuration → latent map**
is separable; freeze-time asks whether the **dynamics-mediated edit** superposes. For the counterfactual version
the baseline is `CF(true configuration)` rather than the real `h0`, so all four states come from the identical
pipeline and the *only* thing that varies is the target configuration.

**Readouts.** Cosine and relative residual `‖composed − direct‖ / ‖direct‖` between the composed and the directly
constructed Δh — and the **Edit Index of the composed state** itself: whether `h_base + Δ_composed`
actually renders the intended scene. A composition can look aligned in state space and still not work — but see
**§7b** before reading that Edit Index as independent evidence, because an affine decoder forces it.

In [ ]:
# [12] §7 — compositionality: build the sequential-composition and object-superposition states.
from editability_metrics import _index_from

COMP_N = 96
CIDX = np.arange(COMP_N)
c_pre = edits.positions[CIDX, ef-1, :N_OBJ, :].astype(np.float32)     # both objects, pre-edit
c_oe  = edits.edit_object[CIDX].astype(int)
c_vel = VEL_ALL[CIDX, ef, :N_OBJ, :].astype(np.float32)
c_obs = edits.obs[CIDX].astype(np.float32)
DT = float(sim["dt"])

def cf_history(targets):
    # counterfactual history: EVERY object on a constant-velocity line arriving at its target at ef
    t_idx = np.arange(ef)
    hist = targets[:, None, :, :] - c_vel[:, None, :, :] * (ef - t_idx)[None, :, None, None] * DT
    return np.stack([render_traj(hist[i]) for i in range(len(hist))]).astype(np.float32)

def ft_frames(starts, targets, n_frames):
    # freeze-time frames: EVERY object interpolates linearly from its start to its target
    fr = np.zeros((len(starts), n_frames, N_OBJ, 2), np.float32)
    for j in range(n_frames):
        fr[:, j] = starts + ((j+1)/n_frames) * (targets - starts)
    return np.stack([render_traj(fr[i], OBS_NOISE) for i in range(len(fr))]).astype(np.float32)

def render_cfg_at(positions):
    return np.stack([render_traj(positions[i][None])[0] for i in range(len(positions))]).astype(np.float32)

def index_vs(pred, gt_edit, gt_uned):
    return _index_from(pred, gt_edit, gt_uned, np.abs(gt_edit - gt_uned) > 1e-3)

# ── targets. (a) an intermediate p1 and a final p2 for the edited object; (b) a move for EACH object ──
rng = np.random.default_rng(0)
d_half = np.array([+1.5, +1.5], np.float32)
p0 = c_pre[np.arange(COMP_N), c_oe]
p2 = edits.positions[CIDX, ef, :N_OBJ, :].astype(np.float32)[np.arange(COMP_N), c_oe]   # the dataset teleport
# The waypoint must lie OFF the straight line p0->p2. A midpoint waypoint would make the two-step
# route traverse EXACTLY the same frames as the 16-frame direct interpolation (verified: max
# difference 0.0), so the test would be vacuous by construction. We offset the midpoint
# perpendicular to p0->p2 so the two routes genuinely take different paths to the same endpoint.
_seg = p2 - p0
_perp = np.stack([-_seg[:, 1], _seg[:, 0]], 1)
_perp = _perp / np.maximum(np.linalg.norm(_perp, axis=1, keepdims=True), 1e-6)
DETOUR = 2.0                                                                             # sim units
p1 = 0.5 * (p0 + p2) + DETOUR * _perp                                                    # detour waypoint
ok_seq = in_frustum(p1) & in_frustum(p2)

delta0 = np.array([+2.0, +1.0], np.float32); delta1 = np.array([-2.0, +1.0], np.float32)
q = c_pre[:, :, :].copy()
q[:, 0] = c_pre[:, 0] + delta0
q[:, 1] = c_pre[:, 1] + delta1
ok_sup = in_frustum(q[:, 0]) & in_frustum(q[:, 1])
print(f"sequential-composition samples: {ok_seq.sum()} of {COMP_N} (waypoint + final both in frustum)")
print(f"object-superposition samples:   {ok_sup.sum()} of {COMP_N} (both moved objects in frustum)")

# configurations for the superposition test: baseline, obj0 only, obj1 only, both
cfg_base = c_pre.copy() + c_vel * DT          # the true continuation into frame ef (nothing edited)
cfg_A = cfg_base.copy(); cfg_A[:, 0] = q[:, 0]
cfg_B = cfg_base.copy(); cfg_B[:, 1] = q[:, 1]
cfg_AB = cfg_base.copy(); cfg_AB[:, 0] = q[:, 0]; cfg_AB[:, 1] = q[:, 1]
GT_BASE, GT_A, GT_B, GT_AB = (render_cfg_at(c) for c in (cfg_base, cfg_A, cfg_B, cfg_AB))
print("rendered the 4 configurations for the superposition test")

In [ ]:
# [13] §7 — run both compositionality tests on both models.
COMP = {}
for name, m in MODELS.items():
    h0c = warm(m, c_obs, ef)
    R_ = {}

    # ---- (a) sequential composition, freeze-time, MATCHED frame counts (16 both routes) ----
    starts = np.repeat(c_pre[:, None, :, :], 1, axis=1)[:, 0]
    tg_direct = cfg_base.copy(); tg_direct[np.arange(COMP_N), c_oe] = p2
    tg_way    = cfg_base.copy(); tg_way[np.arange(COMP_N), c_oe]    = p1
    h_direct = continue_from(m, h0c, ft_frames(c_pre, tg_direct, 16))
    h_step1  = continue_from(m, h0c, ft_frames(c_pre, tg_way, 8))
    h_two    = continue_from(m, h_step1, ft_frames(tg_way, tg_direct, 8))
    R_["sequential (freeze-time)"] = dict(
        d_direct=(h_direct - h0c), d_comp=(h_two - h0c), base=h0c,
        gt_edit=render_cfg_at(tg_direct), gt_uned=GT_BASE, mask=ok_seq)

    # ---- (b) object superposition, BOTH mechanisms ----
    # counterfactual: baseline is CF(true configuration) so all four come from one pipeline
    cf_base = warm(m, cf_history(cfg_base), ef)
    cf_A, cf_B, cf_AB = (warm(m, cf_history(c), ef) for c in (cfg_A, cfg_B, cfg_AB))
    R_["superposition (counterfactual)"] = dict(
        d_direct=(cf_AB - cf_base), d_comp=((cf_A - cf_base) + (cf_B - cf_base)), base=cf_base,
        h_A=cf_A, h_B=cf_B, gt_edit=GT_AB, gt_uned=GT_BASE, mask=ok_sup)
    # freeze-time: all three edits start from the real pre-edit state
    ft_A, ft_B, ft_AB = (continue_from(m, h0c, ft_frames(c_pre, c, N_FT)) for c in (cfg_A, cfg_B, cfg_AB))
    R_["superposition (freeze-time)"] = dict(
        d_direct=(ft_AB - h0c), d_comp=((ft_A - h0c) + (ft_B - h0c)), base=h0c,
        h_A=ft_A, h_B=ft_B, gt_edit=GT_AB, gt_uned=GT_BASE, mask=ok_sup)

    for k, v in R_.items():
        msk = v["mask"]
        dd, dc, bs = v["d_direct"][msk], v["d_comp"][msk], v["base"][msk]
        v["cos"] = float(torch.nn.functional.cosine_similarity(dc, dd, dim=-1).mean())
        v["resid"] = float((torch.linalg.norm(dc - dd, dim=-1) /
                            torch.linalg.norm(dd, dim=-1).clamp_min(1e-9)).mean())
        v["mag_ratio"] = float((torch.linalg.norm(dc, dim=-1) /
                                torch.linalg.norm(dd, dim=-1).clamp_min(1e-9)).mean())
        ge, gu = v["gt_edit"][msk], v["gt_uned"][msk]
        v["idx_direct"] = index_vs(roll(m, bs + dd)[:, 0], ge, gu)
        v["idx_comp"]   = index_vs(roll(m, bs + dc)[:, 0], ge, gu)
        v["idx_base"]   = index_vs(roll(m, bs)[:, 0], ge, gu)
    COMP[name] = R_

rows = ["| model | test | cos(composed, direct) | relative residual ↓ | ‖composed‖/‖direct‖ | Edit Index: unedited | direct | **composed** |",
        "|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    for k, v in COMP[name].items():
        rows.append(f"| {name} | {k} | **{v['cos']:+.3f}** | {v['resid']:.2f} | {v['mag_ratio']:.2f} | "
                    f"{v['idx_base']:+.2f} | {v['idx_direct']:+.2f} | **{v['idx_comp']:+.2f}** |")
display(Markdown("**Table 7 — do edits compose?** *relative residual* = ‖composed − direct‖ / ‖direct‖ (0 = perfect "
                 "composition, ≈1.4 = unrelated directions of equal size). The Edit Index columns apply each state "
                 "and roll it out: `unedited` is the floor, `direct` is what the oracle achieves, **`composed`** is "
                 "whether the sum actually works.\n\n" + "\n".join(rows)))

In [ ]:
# [14] Fig 7 — compositionality: agreement of the displacement, and whether the composed state actually works.
plt.style.use("default")
tests = list(COMP["GRU"].keys())
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
xi = np.arange(len(tests)); w = 0.38
for k, name in enumerate(MODELS):
    ax[0].bar(xi + (k-0.5)*w, [COMP[name][t]["cos"] for t in tests], w,
              color=["#0072B2", "#E69F00"][k], label=name)
    ax[1].bar(xi + (k-0.5)*w, [COMP[name][t]["resid"] for t in tests], w,
              color=["#0072B2", "#E69F00"][k], label=name)
ax[0].axhline(1.0, color="0.35", ls=":", lw=1.3)
ax[0].annotate("perfect composition", xy=(len(tests)-0.45, 1.0), fontsize=7.5, color="0.35", ha="right", va="bottom")
ax[0].axhline(0, color="0.3", lw=0.8); ax[0].set_ylim(-0.1, 1.1)
ax[0].set_ylabel("cos(composed, direct)"); ax[0].set_title("(a) does the sum point the right way?", fontsize=10)
ax[1].axhline(0.0, color="0.35", ls=":", lw=1.3)
ax[1].annotate("perfect composition", xy=(len(tests)-0.45, 0.02), fontsize=7.5, color="0.35", ha="right", va="bottom")
ax[1].set_ylabel("‖composed − direct‖ / ‖direct‖"); ax[1].set_title("(b) how far off is it?", fontsize=10)
for a_ in ax[:2]:
    a_.set_xticks(xi); a_.set_xticklabels(tests, fontsize=8, rotation=20, ha="right")
    a_.legend(fontsize=8); a_.grid(alpha=0.3, axis="y"); style_ax(a_)

w3 = 0.26
for k, (lab, key, col) in enumerate([("unedited (floor)", "idx_base", "0.55"),
                                     ("direct edit (oracle)", "idx_direct", "#009E73"),
                                     ("COMPOSED", "idx_comp", "#D55E00")]):
    for j, name in enumerate(MODELS):
        off = (k-1)*w3 + (j-0.5)*0.09
        ax[2].bar(xi + off, [COMP[name][t][key] for t in tests], 0.085,
                  color=col, alpha=1.0 if j == 0 else 0.55,
                  label=f"{lab} ({name})" if True else None)
for y, lb in [(1.0, "edited world"), (0.0, "equidistant"), (-1.0, "unedited world")]:
    ax[2].axhline(y, color="0.4", ls=":", lw=1.0)
ax[2].set_ylim(-1.05, 1.05); ax[2].set_xticks(xi)
ax[2].set_xticklabels(tests, fontsize=8, rotation=20, ha="right")
ax[2].set_ylabel("Edit Index"); ax[2].set_title("(c) DOES THE COMPOSED STATE WORK?", fontsize=10)
ax[2].legend(fontsize=6.5, ncol=2); ax[2].grid(alpha=0.3, axis="y"); style_ax(ax[2])
fig.suptitle("Fig 7 — compositionality: sequential composition (path-independence) and object superposition",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig7_compositionality.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

for name in MODELS:
    print(f"\n--- {name} ---")
    for t, v in COMP[name].items():
        gain_direct = v["idx_direct"] - v["idx_base"]; gain_comp = v["idx_comp"] - v["idx_base"]
        frac = 100 * gain_comp / max(gain_direct, 1e-6)
        print(f"  {t:<32s} cos {v['cos']:+.3f} | residual {v['resid']:.2f} | "
              f"Edit Index {v['idx_base']:+.2f} -> direct {v['idx_direct']:+.2f}, composed {v['idx_comp']:+.2f} "
              f"({frac:.0f}% of the direct edit's gain)")

---
### §7b — Control: how much of the superposition result is *forced* by an affine decoder?

Sevan's objection, and it is a good one. Two identities conspire here, and neither requires any factorisation:

**(i) The GRU decoder is a single `nn.Linear`, so `decode` is affine.** For *any* vectors `d1, d2`,

$$\text{decode}(h_0+d_1+d_2)\;=\;\text{decode}(h_0+d_1)+\text{decode}(h_0+d_2)-\text{decode}(h_0)$$

identically. Since §7 builds the composed state as exactly `base + (h_A-base) + (h_B-base)`, its **step-0 decode is
algebraically determined** — no structure in the latent is needed to produce it.

**(ii) The renderer is additive over non-overlapping objects**, so in observation space

$$\text{obs}(A\text{ moved}) + \text{obs}(B\text{ moved}) - \text{obs}(\text{neither}) \;=\; \text{obs}(\text{both moved})$$

is *also* an identity — the `-obs(neither)` term is precisely what erases the two stale objects (the "erasure"
worry resolves itself). It breaks only where the objects **overlap in ray space** (the renderer takes the nearest
hit, not a sum) or where the arithmetic leaves `[0,1]` and is clipped.

Together, (i)+(ii) would produce a high composed **Edit Index** on the GRU with no object factorisation whatsoever.

**What this control separates.**
- `RMSE(composed decode, affine prediction)` — **for the GRU this must be ≈ 0 by algebra**; measuring it confirms
  the step-0 Edit Index carries no information beyond linearity. **The RSSM decoder is an MLP** (`_stack` with one
  hidden layer + activation) and is *not* subject to identity (i), so there the gap is a real quantity — and the
  question becomes whether its composed Edit Index **beats** what an affine decoder would have delivered anyway.
- `RMSE(GT_A + GT_B - GT_BASE, GT_AB)` — how well identity (ii) holds in this world, i.e. how much occlusion and
  clipping break it.

**What the control does NOT undercut:** the **state-space** readouts (`cos`, `relative residual`) compare
`d_comp` against `d_direct`, where `d_direct` comes from an *independent* render → teacher-force run. Nothing
algebraic forces those to agree, so they remain the load-bearing numbers for the superposition claim.

In [ ]:
# [14b] §7b — CONTROL: separate "forced by an affine decoder" from a genuine result.
# (i) decoder-affinity test, with the model's own error as the reference scale for the gap.
rows = ["| model | mechanism | RMSE(composed decode, affine prediction) | RMSE(composed decode, GT) — reference scale | Edit Index composed | Edit Index of the affine prediction | Edit Index direct |",
        "|---|---|---|---|---|---|---|"]
for name, m in MODELS.items():
    for mech in ["superposition (counterfactual)", "superposition (freeze-time)"]:
        v = COMP[name][mech]; msk = v["mask"]
        base, hA, hB = v["base"][msk], v["h_A"][msk], v["h_B"][msk]
        pred_comp   = roll(m, base + ((hA - base) + (hB - base)))[:, 0]   # what §7 scores
        pred_affine = roll(m, hA)[:, 0] + roll(m, hB)[:, 0] - roll(m, base)[:, 0]  # forced if decode is affine
        ge, gu = v["gt_edit"][msk], v["gt_uned"][msk]
        gap  = float(np.sqrt(((pred_comp - pred_affine) ** 2).mean()))
        err  = float(np.sqrt(((pred_comp - ge) ** 2).mean()))
        rows.append(f"| {name} | {mech.replace('superposition ','')} | {gap:.2e} | {err:.3f} | "
                    f"{index_vs(pred_comp, ge, gu):+.3f} | {index_vs(pred_affine, ge, gu):+.3f} | "
                    f"{v['idx_direct']:+.3f} |")
display(Markdown("**Table 7b(i) — is the composed *decode* forced?** A single `nn.Linear` decoder makes columns 3 "
                 "and 5 identical by algebra.\n\n" + "\n".join(rows)))

# (ii) does the renderer itself compose additively in this world?
msk_s = ok_sup
id_pred = GT_A[msk_s] + GT_B[msk_s] - GT_BASE[msk_s]
id_rmse = float(np.sqrt(((id_pred - GT_AB[msk_s]) ** 2).mean()))
id_clip = float(np.sqrt(((np.clip(id_pred, 0, 1) - GT_AB[msk_s]) ** 2).mean()))
overlap = float(np.mean(((GT_A[msk_s] > 1e-3) & (GT_B[msk_s] > 1e-3)).any(axis=-1)))
display(Markdown(
    f"**Observation-space identity (model-free).** `RMSE(GT_A + GT_B - GT_BASE, GT_AB)` = **{id_rmse:.4f}** "
    f"(**{id_clip:.4f}** after clipping to [0,1]); reference scale: RMS of `GT_AB` = "
    f"{float(np.sqrt((GT_AB[msk_s]**2).mean())):.4f}. Samples where the two objects share at least one ray "
    f"(where the additive identity must break): **{100*overlap:.0f}%**.  \n"
    f"Edit Index of the pure observation-space identity: **{index_vs(id_pred, GT_AB[msk_s], GT_BASE[msk_s]):+.3f}** "
    f"— this is the ceiling a *perfect* model would score by obeying identity (ii) alone."))

# (iii) the state-space cosine is NOT forced by the decoder — but it needs its own floor.
#      Shuffled controls break the sample-to-sample correspondence while keeping the population of deltas.
g = torch.Generator().manual_seed(0)
rows = ["| model | test | cos(composed, direct) | angle | shuffled floor: other sample's composed | shuffled floor: object B from another sample |",
        "|---|---|---|---|---|---|"]
for name in MODELS:
    for mech, v in COMP[name].items():
        msk = v["mask"]
        dd, dc, bs = v["d_direct"][msk], v["d_comp"][msk], v["base"][msk]
        cs = lambda a, b: float(torch.nn.functional.cosine_similarity(a, b, dim=-1).mean())
        perm = torch.randperm(dd.shape[0], generator=g)
        shuf_all = cs(dc[perm], dd)
        if "h_A" in v:   # only the superposition tests have separable per-object deltas
            dA, dB = v["h_A"][msk] - bs, v["h_B"][msk] - bs
            shuf_B = f"{cs(dA + dB[perm], dd):+.3f}"
        else:
            shuf_B = "n/a — sequential composition has no per-object split"
        rows.append(f"| {name} | {mech} | **{v['cos']:+.3f}** | {np.degrees(np.arccos(np.clip(v['cos'],-1,1))):.0f}° | "
                    f"{shuf_all:+.3f} | {shuf_B} |")
display(Markdown("**Table 7b(iii) — is the *state-space* agreement specific to the matched sample?** Nothing "
                 "algebraic forces these; the shuffled columns are the floor for \"any edit delta resembles any "
                 "other\". Angle is reported because a cosine of +0.87 is a 29° separation, not a 87% match.\n\n"
                 + "\n".join(rows)))

**Current results (updated 2026-08-05).**

- **The decode-level superposition result on the GRU is an artifact.** `RMSE(composed decode, affine prediction)`
  = **6.6e-08** — machine precision. The composed Edit Index (**+0.46**) is *algebraically identical* to
  `decode(h_A) + decode(h_B) − decode(base)` and required no latent structure. It is also close to the **+0.511**
  ceiling that the model-free observation identity scores on its own. Sevan's objection is correct: this column
  carries no information beyond "each single edit works" (already reported as `direct`) plus decoder linearity.
- **The RSSM does not rescue it.** Its MLP decoder genuinely departs from affine (gap **0.101–0.106**, versus its
  own total error 0.195–0.310), but the composed Edit Index (**+0.391** counterfactual) lands *below* the affine
  prediction (**+0.456**). The nonlinearity is a tax, not a bonus — so there is no positive residual attributable
  to object structure on either model.
- **The additive-render identity is real but leaky**: it fails at RMSE 0.177 against an RMS signal of 0.306,
  because the two objects share rays in **42%** of samples (the renderer takes the nearest hit, it does not sum).
  That is why the identity ceiling is +0.51 and not +1.0 — and why the composed Edit Index sits *below* `direct`.
- **The state-space result survives, and is now the load-bearing one.** `cos(composed, direct)` = **+0.873**
  (GRU, counterfactual) against a shuffled floor of **+0.059**, so the agreement is strongly sample-specific.
  The sharper control — keeping object A's true delta but taking object B's from a different sample — scores
  **+0.378**, so roughly half the alignment comes from the correct *pairing*, not from having a generically
  edit-shaped vector. Read it as graded, not binary: +0.873 is a **29° separation**, and the relative residual is
  still **0.52** of the direct edit's magnitude.

**Net:** the claim "edits superpose across objects" holds as a statement about **displacement directions in the
latent**, with a real but partial margin over its controls. It does **not** hold as a statement about the rendered
output, where it was measuring the decoder.

### §7c — What the two-object edit actually looks like

Table 7 says the composed edit recovers most of the direct edit's effect **in state space**; §7b shows the
matching Edit Index is largely forced by the decoder. This is what the composition actually looks like in
observation space — the check that neither number substitutes for. Columns, left to right: the **simulator's ground truth** for the both-moved world; the **unedited** rollout;
the two **single-object** edits on their own; the **directly constructed** both-moved edit; and the **composed**
state `h_base + Δ(obj0) + Δ(obj1)` — the sum of two edits that were each built independently.

Samples are drawn at **random** (seed 0), not cherry-picked for large displacements.

Mechanism shown is the **counterfactual overwrite**, the memoryless one, so the comparison isolates the
configuration → latent map. Because that mechanism *fabricates* a history for every configuration, the context
frames above the edit line are the shared **unedited (baseline) history** — they orient the reader in the scene;
every column's own pre-edit history is its own fabrication, and the honest comparison is everything below the line,
which is each column's own free-run.

In [ ]:
# [15] Fig 8 — observation waterfalls for the two-object (superposition) edit, canonical spec.
N_CTX_C = 6
DARK_C, TXT_C, TICK_C, EDIT_C2 = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TGT_C, GHO_C = "#00E676", "#FF5252"

def render_cfg_ids(positions):
    # single-frame clean render of a configuration, returning intensity AND per-ray object id
    ints, ids = [], []
    for i in range(len(positions)):
        sc = Scene(positions=positions[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                   radii=RAD, colors=COL, reflectivities=REFL, config=_cfg(1, 0.0))
        _, rid, rint = render_scene(sc); ints.append(rint[0]); ids.append(rid[0])
    return np.stack(ints).astype(np.float32), np.stack(ids).astype(np.int64)

# GT for the both-moved world, rolled FORWARD K steps (each object continues with its own velocity)
gt_ab_traj = np.zeros((COMP_N, K_ROLL, R), np.float32)
for s_ in range(K_ROLL):
    gt_ab_traj[:, s_] = render_cfg_ids(cfg_AB + c_vel * DT * s_)[0]
_, ids_AB   = render_cfg_ids(cfg_AB)
_, ids_BASE = render_cfg_ids(cfg_base)
ctx_cf = cf_history(cfg_base)[:, ef-N_CTX_C:ef, :]      # shared unedited (baseline) history

def _cx2(mask_row):
    i = np.where(mask_row)[0]; return i.mean() if i.size else np.nan
tgt_cx2 = [np.array([_cx2(ids_AB[i] == k) for i in range(COMP_N)]) for k in (0, 1)]
gho_cx2 = [np.array([_cx2((ids_BASE[i] == k) & (ids_AB[i] != k)) for i in range(COMP_N)]) for k in (0, 1)]

rng_w = np.random.default_rng(0)
SAMP_C = list(rng_w.choice(np.where(ok_sup)[0], size=3, replace=False))
print("random superposition samples for the waterfall:", SAMP_C)

def waterfall_comp(model_name, mech="superposition (counterfactual)"):
    m = MODELS[model_name]; v = COMP[model_name][mech]
    cols = ["GT (sim)\nboth objects moved", "unedited\n(no edit)", "move obj0 only", "move obj1 only",
            "DIRECT\nmove both", "COMPOSED\nΔ(obj0) + Δ(obj1)"]
    bodies = [gt_ab_traj,
              roll(m, v["base"]), roll(m, v["h_A"]), roll(m, v["h_B"]),
              roll(m, v["base"] + v["d_direct"]), roll(m, v["base"] + v["d_comp"])]
    fig, axes = plt.subplots(len(SAMP_C), len(cols), figsize=(3.0*len(cols), 3.4*len(SAMP_C)),
                             squeeze=False, facecolor=DARK_C)
    for r_, smp in enumerate(SAMP_C):
        for c_ in range(len(cols)):
            ax_ = axes[r_][c_]; ax_.set_facecolor(DARK_C)
            panel = np.clip(np.concatenate([ctx_cf[smp], bodies[c_][smp]], 0), 0, 1)
            ax_.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                       interpolation="nearest")
            for sp in ax_.spines.values(): sp.set_edgecolor(TICK_C)
            ax_.axhline(N_CTX_C-0.5, color=EDIT_C2, lw=1.4, ls="--", alpha=0.95)
            for k in (0, 1):
                if not np.isnan(tgt_cx2[k][smp]): ax_.axvline(tgt_cx2[k][smp], color=TGT_C, lw=1.5, alpha=0.9)
                if not np.isnan(gho_cx2[k][smp]): ax_.axvline(gho_cx2[k][smp], color=GHO_C, ls="--", lw=1.5, alpha=0.9)
            if r_ == 0: ax_.set_title(cols[c_], fontsize=8, color=TXT_C)
            if c_ == 0:
                ax_.set_ylabel(f"sample {smp}\nsim frame", fontsize=8, color=TXT_C)
                ax_.set_yticks([0, N_CTX_C, N_CTX_C+7, N_CTX_C+14])
                ax_.set_yticklabels([ef-N_CTX_C, ef, ef+7, ef+14], fontsize=7)
            else: ax_.set_yticks([])
            ax_.set_xlabel("ray", fontsize=8, color=TXT_C); ax_.tick_params(colors=TICK_C, labelsize=7)
    handles = [Line2D([0],[0], color=TGT_C, lw=2.2, label="target locations (BOTH objects)"),
               Line2D([0],[0], color=GHO_C, ls="--", lw=2.2, label="ghost / vacated locations (both objects)"),
               Line2D([0],[0], color=EDIT_C2, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX_C} shared unedited-history frames above; every row below "
                            f"is that column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT_C, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(f"Fig 8{'ab'[list(MODELS).index(model_name)]} — {model_name}: does the SUM of two single-object "
                 f"edits reproduce the both-objects edit?  ({mech})", y=1.0, fontsize=10.5, color=TXT_C)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fn = f"fig8{'ab'[list(MODELS).index(model_name)]}_superposition_waterfall_{model_name}.png"
    fig.savefig(f"{OUT}/{fn}", dpi=130, bbox_inches="tight", facecolor=DARK_C)
    display(fig); plt.close(fig); print("saved", fn)

for nm in MODELS:
    waterfall_comp(nm)

---
## §8 — Summary

In [ ]:
# [16] Computed summary.
print("================ SUMMARY (computed, not asserted) ================")
for name in MODELS:
    u = CARDS[name]["unsteered (no edit)"]["edit_index"]
    print(f"\n--- {name} (unsteered Edit Index {u:+.2f} = its own '−1' end) ---")
    print(f"1. DO THE ORACLES SUCCEED?  counterfactual {CARDS[name]['counterfactual state overwrite']['edit_index']:+.2f}"
          f" | freeze-time {CARDS[name]['freeze-time teacher forcing']['edit_index']:+.2f}"
          f" | readout injection {CARDS[name]['readout injection (the failing editor)']['edit_index']:+.2f}")
    ci = CARDS[name]['counterfactual state overwrite']['edit_index_by_step']
    fi = CARDS[name]['freeze-time teacher forcing']['edit_index_by_step']
    print(f"   persistence (step 0 -> 14): counterfactual {ci[0]:+.2f} -> {ci[-1]:+.2f} | "
          f"freeze-time {fi[0]:+.2f} -> {fi[-1]:+.2f}")
    P = PROBES[name]["position (d=4)"]; ch = np.sqrt(P["d"]/MODELS[name].hidden_size)
    fc = RF[name]["counterfactual — raw"]["position (d=4)"][0]
    ff = RF[name]["freeze-time — raw"]["position (d=4)"][0]
    print(f"2. REACHABILITY CEILING (position probe, chance {ch:.3f}): counterfactual {fc:.3f} ({fc/ch:.1f}x chance)"
          f" | freeze-time {ff:.3f} ({ff/ch:.1f}x chance)")
    print(f"   => a readout-injection edit could match at best {100*max(fc,ff):.0f}% of a successful edit's direction.")
    fc8 = RF[name]["counterfactual — raw"]["position+velocity (d=8)"][0]
    ch8 = np.sqrt(PROBES[name]["position+velocity (d=8)"]["d"]/MODELS[name].hidden_size)
    print(f"   adding velocity to the probe: {fc:.3f} -> {fc8:.3f} (chance {ch:.3f} -> {ch8:.3f}) — "
          + ("the extra readout DOES capture more" if fc8/ch8 > fc/ch + 0.3 else
             "no real gain once chance is accounted for: the missing content is NOT physical state"))
    rnd = 1/np.sqrt(MODELS[name].hidden_size)
    print(f"3. DO THE ORACLES AGREE? cos(counterfactual, freeze-time) = "
          f"{COS[name]['counterfactual vs freeze-time (raw)']:+.3f} raw, "
          f"{COS[name]['counterfactual vs freeze-time (edit-only)']:+.3f} edit-only "
          f"(shuffled control {COS[name]['SHUFFLED control (cf vs ft, mismatched)']:+.3f}, random {rnd:+.3f})")
    print(f"   vs the failing editor: cos(counterfactual, readout injection) = "
          f"{COS[name]['counterfactual vs readout injection']:+.3f}")
    q = MAG[name]["counterfactual — raw"]
    print(f"4. MAGNITUDE: ‖Δh‖/‖h0‖ {q['‖Δh‖ / ‖h0‖']:.2f} | vs the injection it replaces "
          f"{q['‖Δh_true‖ / ‖Δh_pinv‖']:.2f}x | vs ONE ordinary dynamics step "
          f"{q['‖Δh‖ / mean‖h_t−h_{t−1}‖']:.1f}x")
    print(f"5. CONSISTENCY: mean pairwise cosine across edits "
          f"{q['direction consistency (mean pairwise cos)']:+.3f} "
          f"(expected 0 for unrelated directions; per-pair sd 1/sqrt(H) = {rnd:.3f}) | "
          f"magnitude CV {q['magnitude CV']:.2f} -> "
          + ("a shared edit direction exists" if q['direction consistency (mean pairwise cos)'] > 0.2
             else "no shared direction: each edit has its own"))
    best = max(LEARNED[name], key=lambda k: LEARNED[name][k]["edit_index"])
    print(f"6. LEARNED FROM ORACLE Δh: best = {best} | train R² {LEARNED[name][best]['train_r2']:.3f} vs "
          f"HELD-OUT R² {LEARNED[name][best]['test_r2']:+.3f} (cos {LEARNED[name][best]['test_cos']:+.3f}) "
          f"-> {'MEMORISED (fits train, not held-out)' if LEARNED[name][best]['test_r2'] < 0.5*LEARNED[name][best]['train_r2'] else 'generalises on Δh itself'}")
    print(f"   applied: Edit Index {LEARNED[name][best]['edit_index']:+.2f} "
          f"(unsteered {u:+.2f}, oracle it imitates "
          f"{max(CARDS[name]['counterfactual state overwrite']['edit_index'], CARDS[name]['freeze-time teacher forcing']['edit_index']):+.2f})")
print("\nPNGs:", sorted(os.listdir(OUT)))